# Penjelasan Notebook: Perbandingan Pandas vs Dask pada Data Besar
Notebook ini membuat **5 juta baris** data acak karyawan, menyimpannya ke CSV, lalu membandingkan cara membaca dan mengolahnya menggunakan **Pandas** (eager, memuat semua ke memori) dan **Dask** (lazy, diproses bertahap dan paralel).

**Poin penting:** Dask bersifat *lazy*. Waktu baca 0,02 detik bukan berarti lebih cepat, karena data belum benar-benar dibaca. Gunakan `.compute()` untuk mendapatkan hasil sebenarnya.

## Cell 1: Menentukan jumlah baris data
- `num_row = 5_000_000` menyimpan angka **5 juta** ke variabel `num_row`.
- Garis bawah `_` hanya pemisah ribuan agar mudah dibaca (sama dengan `5000000`).
- Variabel ini dipakai untuk menentukan ukuran semua data yang dibuat di cell berikutnya.

In [7]:
# Jumlah baris data yang akan dibuat (5 juta). _ hanya pemisah ribuan, tidak mengubah nilai
num_row = 5_000_000

## Cell 2: Import library
- `import dask.dataframe as dd` : library **Dask** untuk mengolah data berukuran besar secara bertahap (*lazy*). Diberi alias `dd`.
- `import pandas as pd` : library **Pandas** untuk tabel data (DataFrame). Alias `pd`.
- `import numpy as np` : library **NumPy** untuk operasi angka dan array acak. Alias `np`.
- `import time` : modul bawaan Python untuk mengukur waktu eksekusi.

In [8]:
# Import library yang dibutuhkan: Dask (data besar), Pandas (tabel), NumPy (angka), time (ukur waktu)
import dask.dataframe as dd
import pandas as pd
import numpy as np
import time

## Cell 3: Membuat data acak (dictionary)
Kode ini membuat **dictionary** Python, di mana setiap *key* adalah nama kolom dan *value*-nya adalah array berisi 5 juta nilai.

- `"id": np.arange(num_row)` : angka berurutan 0 sampai 4.999.999.
- `np.random.choice([...], size=num_row)` : memilih nama/departemen secara acak dari daftar yang diberikan.
- `np.random.randint(18, 70, size=num_row)` : angka acak bulat antara 18 dan 69 (untuk `age`).
- `np.random.randint(30_000, 150_000, ...)` : gaji acak antara 30.000 dan 149.999.
- `pd.date_range("2010-01-01", "2022-12-31")` : membuat daftar **semua tanggal** di rentang tersebut, lalu `np.random.choice` memilih tanggal acak untuk `join_date`.

In [9]:
# Membuat dictionary berisi 6 kolom, masing-masing berisi 5 juta data acak
# id: nomor urut | name: nama acak | age: umur acak | salary: gaji acak
# department: departemen acak | join_date: tanggal bergabung acak
data = {
    "id": np.arange(num_row),
    "name": np.random.choice(["Alice", "Bob", "Charlie", "Diana"], size=num_row),
    "age": np.random.randint(18, 70, size=num_row),
    "salary": np.random.randint(30_000, 150_000, size=num_row),
    "department": np.random.choice(["HR", "IT", "Finance", "Marketing"], size=num_row),
    "join_date": np.random.choice(pd.date_range("2010-01-01", "2022-12-31"), size=num_row)
}

## Cell 4: Mengubah dictionary menjadi DataFrame
- `pd.DataFrame(data)` mengubah dictionary menjadi **tabel** (baris dan kolom) berbentuk DataFrame Pandas.
- Hasilnya disimpan ke variabel `df`.

In [10]:
# Ubah dictionary menjadi tabel (DataFrame) Pandas
df = pd.DataFrame(data)

## Cell 5: Menyimpan data ke file CSV
- `df.to_csv('large_Dataset.csv', index=False)` : menyimpan tabel ke file CSV bernama `large_Dataset.csv`.
  - `index=False` berarti nomor index (0, 1, 2, ...) **tidak** ikut disimpan sebagai kolom tambahan.
- `print(...)` : menampilkan pesan bahwa proses berhasil.

In [11]:
# Simpan DataFrame ke file CSV tanpa kolom index
# Tampilkan pesan konfirmasi
df.to_csv('large_Dataset.csv',index=False)
print('Dataset Anda Berhasil....')

Dataset Anda Berhasil....


## Cell 6: Membaca CSV dengan Pandas dan mengukur waktunya
- `time.time()` mencatat waktu **sebelum** dan **sesudah** proses.
- `pd.read_csv(...)` membaca **seluruh** file CSV ke memori sekaligus (*eager*).
- `end - start` adalah selisih waktu = lama proses.
- `:.2f` pada f-string membatasi tampilan menjadi 2 angka di belakang koma.

Hasil: sekitar **4,87 detik**.

In [20]:
# Catat waktu mulai
# Baca seluruh file CSV ke memori (langsung dimuat semua)
# Catat waktu selesai
# Tampilkan durasi, dibulatkan 2 desimal
start = time.time()
df = pd.read_csv('/content/large_Dataset.csv')
end = time.time()

print(f'Hasil waktu pandas : {end - start:.2f} seconds')

Hasil waktu pandas : 4.87 seconds


## Cell 7: Membaca CSV dengan Dask dan mengukur waktunya
- `dd.read_csv(...)` membuat **Dask DataFrame**. Dask **tidak langsung** membaca seluruh data; ia hanya membagi file menjadi beberapa *partition* dan menyiapkan rencana kerja (*lazy evaluation*).
- Itulah sebabnya waktunya sangat kecil, sekitar **0,02 detik**. Ini **bukan** berarti Dask lebih cepat membaca, melainkan data belum benar-benar dibaca.
- Data baru diproses ketika memanggil `.compute()`.

In [21]:
# Buat Dask DataFrame (lazy: hanya menyiapkan rencana, data belum dibaca penuh)
# Catatan: waktu kecil karena belum ada proses sebenarnya; gunakan .compute() untuk hasil nyata
start = time.time()
df_dask = dd.read_csv('/content/large_Dataset.csv')
end = time.time()

print(f'Hasil waktu dask : {end - start:.2f} seconds')

Hasil waktu dask : 0.02 seconds


## Cell 8: Menampilkan DataFrame Pandas
- Menulis nama variabel `df` saja di baris terakhir sel akan menampilkan isi tabel.
- Output menampilkan **5 juta baris**, tetapi Jupyter/Colab hanya menampilkan sebagian (awal dan akhir) agar tidak berat.

In [23]:
# Menampilkan isi DataFrame Pandas (otomatis ditampilkan sebagian)
df

,id,name,age,salary,department,join_date
0,0,Charlie,28,136090,IT,2019-06-23
1,1,Alice,19,75374,IT,2016-06-22
2,2,Bob,56,34010,IT,2021-10-21
3,3,Bob,22,102744,IT,2017-09-20
4,4,Bob,60,123767,Finance,2016-03-27
...,...,...,...,...,...,...
4999995,4999995,Charlie,28,58336,Marketing,2018-08-29
4999996,4999996,Charlie,26,61991,IT,2021-07-15
4999997,4999997,Bob,23,67446,IT,2021-12-18
4999998,4999998,Charlie,18,130111,Marketing,2010-08-07


## Cell 9: Melihat struktur Dask DataFrame
- Menulis `df_dask` tanpa perintah lain hanya menampilkan **struktur** (nama kolom dan tipe datanya), bukan data.
- `npartitions=3` berarti data dibagi menjadi **3 partition**.
- Tipe data seperti `int64` (angka bulat) dan `string` (teks) terlihat di sini.

In [22]:
# Menampilkan struktur Dask DataFrame (kolom, tipe data, jumlah partition)
df_dask

,id,name,age,salary,department,join_date
npartitions=3,,,,,,
,int64,string,int64,int64,string,string
,...,...,...,...,...,...
,...,...,...,...,...,...
,...,...,...,...,...,...


## Cell 10: `head()` pada Dask
- `df_dask.head()` mengambil **5 baris pertama** saja.
- Dask tidak perlu memproses seluruh data untuk ini, karena cukup mengambil bagian awal.

In [24]:
# Ambil 5 baris pertama untuk melihat contoh data
df_dask.head()

,id,name,age,salary,department,join_date
0,0,Charlie,28,136090,IT,2019-06-23
1,1,Alice,19,75374,IT,2016-06-22
2,2,Bob,56,34010,IT,2021-10-21
3,3,Bob,22,102744,IT,2017-09-20
4,4,Bob,60,123767,Finance,2016-03-27


## Cell 11: Menghitung jumlah baris
- `df_dask.shape[0]` : mengambil jumlah baris (dimensi pertama). Hasilnya masih berupa *rencana* (lazy).
- `.compute()` : menjalankan rencana tersebut dan menghasilkan angka sebenarnya, yaitu **5.000.000**.
- Tanpa `.compute()`, hasilnya hanya berupa objek Dask, bukan angka.

In [25]:
# Hitung jumlah baris; .compute() menjalankan perhitungan sebenarnya
df_dask.shape[0].compute()

5000000

## Cell 12: Rata-rata kolom `salary`
- `df_dask['salary']` : memilih kolom `salary`.
- `.mean()` : menghitung rata-rata. Hasil sekitar **89.987**.
- `.compute()` : menjalankan perhitungan. Dask menghitung rata-rata per partition lalu menggabungkannya.

In [26]:
# Hitung rata-rata gaji dari seluruh data
df_dask['salary'].mean().compute()

np.float64(89987.586265)

## Cell 13: Memfilter data (gaji di atas 100.000)
- `df_dask['salary'] > 100_000` : membuat kondisi **True/False** untuk setiap baris.
- `df_dask[kondisi]` : hanya mengambil baris yang kondisinya True (*filtering*).
- `.head()` : menampilkan 5 baris pertama dari hasil filter.

In [31]:
# Ambil baris yang gajinya lebih dari 100.000, tampilkan 5 teratas
df_dask[df_dask['salary'] > 100_000].head()

,id,name,age,salary,department,join_date
0,0,Charlie,28,136090,IT,2019-06-23
3,3,Bob,22,102744,IT,2017-09-20
4,4,Bob,60,123767,Finance,2016-03-27
5,5,Charlie,36,113036,IT,2015-05-21
11,11,Diana,61,102012,Marketing,2011-07-12


## Cell 14: Mengelompokkan dan menghitung rata-rata (`groupby`)
- `groupby('department')` : mengelompokkan data berdasarkan departemen (Finance, HR, IT, Marketing).
- `['salary'].mean()` : menghitung rata-rata gaji untuk **setiap kelompok**.
- `.compute()` : menjalankan perhitungan.

Hasilnya menunjukkan rata-rata gaji per departemen, dan nilainya hampir sama karena data dibuat secara acak.

In [32]:
# Kelompokkan berdasarkan departemen lalu hitung rata-rata gaji tiap departemen
df_dask.groupby('department')['salary'].mean().compute()

,salary
department,
Finance,90007.917651
HR,89999.340538
IT,89989.008117
Marketing,89954.080471


## Cell 15: Mengurutkan data (`sort_values`)
- `sort_values(by='salary', ascending=False)` : mengurutkan dari gaji **terbesar ke terkecil**.
- `.head()` : mengambil 5 baris teratas, yaitu gaji tertinggi.

Catatan: pengurutan pada data besar cukup berat karena Dask perlu membandingkan seluruh data.

In [33]:
# Urutkan data berdasarkan gaji dari yang tertinggi, ambil 5 teratas
df_dask.sort_values(by='salary', ascending=False).head()

,id,name,age,salary,department,join_date
19655,19655,Bob,69,149999,Marketing,2015-09-30
1239973,1239973,Charlie,28,149999,HR,2013-04-05
190715,190715,Charlie,55,149999,HR,2022-03-02
884735,884735,Bob,37,149999,IT,2022-07-12
1528715,1528715,Charlie,45,149999,IT,2012-06-17


## Cell 16: Filter dengan dua kondisi
- `(kondisi1) & (kondisi2)` : operator `&` berarti **DAN** (keduanya harus benar).
- Di sini diambil karyawan dari departemen **IT** yang gajinya **di atas 120.000**.
- Catatan: setiap kondisi wajib dibungkus tanda kurung `()` saat memakai `&`.

In [36]:
# Ambil karyawan IT dengan gaji di atas 120.000 (kedua syarat harus terpenuhi)
df_dask[(df_dask['department'] == 'IT') & (df_dask['salary'] > 120_000)].head()

,id,name,age,salary,department,join_date
0,0,Charlie,28,136090,IT,2019-06-23
63,63,Diana,32,121119,IT,2017-03-18
68,68,Bob,41,124727,IT,2015-10-15
82,82,Charlie,41,130590,IT,2013-07-30
104,104,Diana,42,137949,IT,2012-06-22


## Cell 17: Menghitung jumlah setiap nilai (`value_counts`)
- `['department'].value_counts()` : menghitung berapa kali setiap departemen muncul.
- Hasilnya terlihat seimbang, sekitar 1,25 juta untuk tiap departemen.
- `.compute()` : menjalankan perhitungan.

In [37]:
# Hitung jumlah karyawan di setiap departemen
df_dask['department'].value_counts().compute()

,count
department,
IT,1249535
Finance,1250662
Marketing,1250279
HR,1249524


## Cell 18: Ringkasan statistik (`describe`)
- `describe()` menampilkan statistik ringkas: `count` (jumlah), `mean` (rata-rata), `std` (standar deviasi), `min`, `25%`, `50%` (median), `75%`, dan `max`.
- `.compute()` diperlukan agar hasil dihitung dan ditampilkan.

In [38]:
# Tampilkan statistik ringkas (jumlah, rata-rata, min, max, kuartil, dan lain-lain)
df_dask.describe().compute()

,id,age,salary
count,5.000000e+06,5.000000e+06,5.000000e+06
mean,2.500000e+06,4.350221e+01,8.998759e+04
std,1.443376e+06,1.500479e+01,3.464577e+04
min,0.000000e+00,1.800000e+01,3.000000e+04
25%,8.424140e+05,3.100000e+01,5.999000e+04
50%,2.099192e+06,4.400000e+01,9.003400e+04
75%,3.342283e+06,5.700000e+01,1.200020e+05
max,4.999999e+06,6.900000e+01,1.499990e+05


## Cell 19: Statistik dengan Pandas (`describe` dan `apply`)
- `df[['id', 'age', 'salary']]` : memilih tiga kolom saja.
- `.describe()` : statistik ringkas untuk kolom-kolom tersebut.
- `.apply(lambda x: x.map(lambda y: f'{y:0f}'))` :
  - `apply` menjalankan fungsi pada setiap kolom.
  - `lambda` adalah fungsi kecil tanpa nama.
  - `map` menjalankan fungsi pada setiap nilai.
  - `f'{y:0f}'` memformat angka menjadi teks *float*.

Catatan: `:0f` tidak membatasi jumlah desimal (default 6 digit). Untuk 2 desimal gunakan `:.2f`.

In [45]:
# Statistik untuk kolom id, age, salary; lalu format angka menjadi teks
# Catatan: :0f menampilkan 6 desimal; gunakan :.2f jika ingin 2 desimal
df[['id', 'age', 'salary']].describe().apply(lambda x: x.map(lambda y: f'{y:0f}'))

,id,age,salary
count,5000000.000000,5000000.000000,5000000.000000
mean,2499999.500000,43.502212,89987.586265
std,1443375.817312,15.004790,34645.771232
min,0.000000,18.000000,30000.000000
25%,1249999.750000,31.000000,59974.000000
50%,2499999.500000,43.000000,90005.000000
75%,3749999.250000,56.000000,119988.000000
max,4999999.000000,69.000000,149999.000000


## Cell 20: Mengambil sampel acak (`sample`)
- `sample(frac=0.001)` : mengambil **0,1%** dari seluruh data secara acak, sekitar 5.000 baris.
- Berguna untuk melihat gambaran data tanpa memproses seluruhnya.
- `.compute()` : menjalankan pengambilan sampel.

In [46]:
# Ambil sampel acak 0,1% dari data (sekitar 5.000 baris)
df_dask.sample(frac=0.001).compute()

,id,name,age,salary,department,join_date
1657389,1657389,Charlie,22,54899,IT,2022-11-25
1474704,1474704,Alice,66,87116,IT,2021-06-22
734866,734866,Alice,54,49369,Finance,2016-09-24
115030,115030,Diana,33,135205,Marketing,2022-09-23
6828,6828,Diana,47,145112,IT,2011-01-23
...,...,...,...,...,...,...
1105258,4447541,Alice,33,83029,Marketing,2022-02-17
338201,3680484,Diana,69,141899,Marketing,2017-09-06
1118114,4460397,Alice,69,115721,IT,2014-05-20
27389,3369672,Alice,63,102360,Marketing,2018-12-11


## Cell 21: Melihat jumlah partition
- `df_dask.npartitions` menampilkan jumlah **partition**, yaitu potongan data yang diproses Dask secara terpisah (paralel).
- Hasilnya `3`, sesuai dengan `npartitions=3` di cell 9.

In [47]:
# Lihat jumlah partition (potongan data) pada Dask
df_dask.npartitions

3

## Cell 22: Mengambil satu partition
- `get_partition(0)` : mengambil **partition ke-0** saja (potongan data pertama) sebagai Pandas DataFrame.
- `.head()` : menampilkan 5 baris pertama dari partition tersebut.
- Berguna untuk memeriksa salah satu bagian data.

In [48]:
# Ambil partition pertama (indeks 0), lalu tampilkan 5 baris awalnya
df_dask.get_partition(0).head()

,id,name,age,salary,department,join_date
0,0,Charlie,28,136090,IT,2019-06-23
1,1,Alice,19,75374,IT,2016-06-22
2,2,Bob,56,34010,IT,2021-10-21
3,3,Bob,22,102744,IT,2017-09-20
4,4,Bob,60,123767,Finance,2016-03-27
